<a href="https://colab.research.google.com/github/cpython-projects/python_da_06_11_25/blob/main/lesson_27_part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import plotly.express as px

In [ ]:
DB_USER = "prog_academy_da_user"
DB_PASS = "4hDTs4hvD9wnplWqDrnl2nSsmYiynpsX"
DB_HOST = "dpg-d28ed56r433s73e3pn9g-a.oregon-postgres.render.com"
DB_PORT = "5432"
DB_NAME = "prog_academy_da"

In [ ]:
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

У нас є користувачі, які:

* встановлюють застосунок (`app_sessions`),
* переглядають товари (`product_views_log`),
* авторизуються (`devices_users_map`),
* здійснюють покупки (`orders_log`).

# Робота з датами у SQL
**Основні функції для роботи з датою при аналізі даних**

## `DATE_TRUNC`  
**обрізає дату або час до заданої одиниці виміру**, обнуляючи менш значущі компоненти.

### Загальний синтаксис:

```sql
DATE_TRUNC('одиниця_часу', timestamp)
```

---

### Для чого використовується:

1. **Групування у часі**
   Наприклад, по тижнях, місяцях, днях тощо.
2. **Округлення до початку години/дня і т.д.**
3. **Порівняння дат з потрібною точністю**
   (наприклад, всі події за конкретний місяць).

---

### Підтримувані одиниці:

* `millennium`, `century`, `decade`, `year`, `quarter`, `month`
* `week`, `day`, `hour`, `minute`, `second`

In [ ]:
query = """
  SELECT DATE_TRUNC('week', first_seen) FROM app_sessions LIMIT 5;
"""

df = pd.read_sql_query(query, engine)
df

,date_trunc
0,2024-04-22 00:00:00+00:00
1,2024-01-15 00:00:00+00:00
2,2024-01-15 00:00:00+00:00
3,2024-02-19 00:00:00+00:00
4,2024-04-22 00:00:00+00:00


## `EXTRACT`
**витягує окрему частину дати або часу**, наприклад: рік, місяць, день, година і т.д.

### Синтаксис:

```sql
EXTRACT(одиниця_часу FROM дата_або_час)
```

---

### Використовувані одиниці:

* `YEAR`
* `MONTH`
* `DAY`
* `HOUR`
* `MINUTE`
* `SECOND`
* `DOW` (day of week: 0 = Sunday, 6 = Saturday)
* `DOY` (day of year)
* `WEEK`
* `QUARTER`

---

### Для чого застосовується:

1. **Фільтрація по частині дати**
2. **Створення часових групувань**

---

### Різниця з `DATE_TRUNC`:

|                | `EXTRACT`                     | `DATE_TRUNC`                                 |
| -------------- | ----------------------------- | -------------------------------------------- |
| Що робить      | Повертає одне число           | Обрізає менш значущі частини дати            |
| Тип результату | `numeric`                     | `timestamp`                                  |
| Приклад        | `EXTRACT(MONTH FROM d)` → `7` | `DATE_TRUNC('month', d)` → `'2025-07-01...'` |

In [ ]:
query = """
SELECT EXTRACT(DOY FROM first_seen) FROM app_sessions LIMIT 5;
"""

df = pd.read_sql_query(query, engine)
df

,extract
0,116.0
1,15.0
2,18.0
3,54.0
4,115.0


## `CURRENT_DATE`, `CURRENT_TIMESTAMP` и `NOW()`
**поточна дата та/або час**

### Особенности

| Функція             | Що повертає                     | Тип даних   | Приклад                      |
| ------------------- | ------------------------------- | ----------- | ---------------------------- |
| `CURRENT_DATE`      | Лише поточну дату               | `DATE`      | `2025-07-14`                 |
| `CURRENT_TIMESTAMP` | Поточну дату **та** час         | `TIMESTAMP` | `2025-07-14 11:25:32.123456` |
| `NOW()`             | Те саме, що `CURRENT_TIMESTAMP` | `TIMESTAMP` | `2025-07-14 11:25:32.123456` |


In [ ]:
query = """
SELECT CURRENT_DATE, CURRENT_TIMESTAMP, NOW();
"""

df = pd.read_sql_query(query, engine)
df

,current_date,current_timestamp,now
0,2025-08-26,2025-08-26 16:23:47.969546+00:00,2025-08-26 16:23:47.969546+00:00


## `INTERVAL`  
**додавання/віднімання часових проміжків** (дні, години, місяці, секунди і т.д.)

### 📌 Синтаксис:

```sql
SELECT CURRENT_DATE + INTERVAL '7 days';
```
---

### Одиниці:

* `second`, `minute`, `hour`
* `day`, `week`, `month`, `year`
* * комбінації, наприклад:

```sql
INTERVAL '2 days 3 hours 15 minutes'
```
---

### Особливості:

* `INTERVAL` сам по собі — не дата, це **часовий відрізок**
* Використовується з `DATE` і з `TIMESTAMP`
* В MySQL запис трохи інший:

```sql
-- MySQL стиль
SELECT NOW() + INTERVAL 1 DAY;
```
---

### В Pandas аналог:

```python
import pandas as pd

pd.Timestamp.now() + pd.Timedelta(days=3, hours=2)
```

In [ ]:
query = """
SELECT CURRENT_DATE + INTERVAL '7 day';
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    result = result.fetchone()
    print(result)

(datetime.datetime(2025, 9, 2, 0, 0),)


In [ ]:
query = """
SELECT CURRENT_DATE + INTERVAL '7 month';
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    result = result.fetchone()
    print(result)

(datetime.datetime(2026, 3, 26, 0, 0),)


In [ ]:
query = """
SELECT CURRENT_DATE + INTERVAL '1 month 2 years 6 days';
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    result = result.fetchone()
    print(result)

(datetime.datetime(2027, 10, 2, 0, 0),)


In [ ]:
query = """
SELECT session_id, first_seen, first_seen + INTERVAL '7 day' AS tomorrow
FROM app_sessions
WHERE first_seen >= CURRENT_DATE - INTERVAL '500 day';
"""

df = pd.read_sql_query(query, engine)
df

,session_id,first_seen,tomorrow
0,S00000,2024-04-25,2024-05-02
1,S00004,2024-04-24,2024-05-01
2,S00005,2024-06-02,2024-06-09
3,S00007,2024-05-25,2024-06-01
4,S00009,2024-05-05,2024-05-12
...,...,...,...
216,S00491,2024-05-28,2024-06-04
217,S00492,2024-04-21,2024-04-28
218,S00494,2024-04-18,2024-04-25
219,S00495,2024-06-20,2024-06-27


## Коли застосовується робота з датами та часом

| Ціль                                      | Коли застосовуємо                                  | Що потрібно                      |
| ----------------------------------------- | -------------------------------------------------- | -------------------------------- |
| **1. Графіки по днях / тижнях / місяцях** | Динаміка показників у часі                         | Групування по даті               |
| **2. Сезонність**                         | Піки/спади у поведінці                             | Групування по дню тижня, місяцю  |
| **3. Повторюваність покупок**             | Як часто повертаються користувачі                  | Обчислення інтервалів між датами |
| **4. Retention**                          | Чи повертаються користувачі після першої взаємодії | Когорти за першою подією         |